# How can MoM-BH\*-1 hold itself together? A physics simulator (stages 1 and 2)

**The question:** MoM-BH\*-1 is a black hole wrapped in a huge envelope of gas. What stops that envelope from falling into the black hole, or being blown away by the black hole's light? And which black-hole and envelope masses would make it look exactly the way you measured?

**Physics this simulator includes:**
- **Gravity vs pressure (hydrostatic equilibrium):** at every radius, the outward push of pressure balances the inward pull of gravity.
- **Mass conservation:** all the layers add up to the total mass.
- **Gas + radiation pressure:** pressure ∝ density^(4/3). That's what happens when radiation pressure dominates and convection churns the envelope (an "n = 3 polytrope").
- **Black hole feeding (Bondi accretion):** inside the Bondi radius, r_B = 2GM_BH / c_s², gas falls into the black hole.
- **Energy:** the black hole's light is limited by how fast convection can carry energy out of the Bondi region, L = 4π r_B² (η/Γ) ρ c_s³, with efficiency η = 0.1 (the prescription of Ball et al. 2011).
- **Eddington limit:** the light must not push harder than gravity pulls.
- **Stefan–Boltzmann law at the surface:** L = 4πR²σT⁴.

**Left out** (list these as limitations): rotation, magnetic fields, 3D turbulence, winds, the way the cool surface layers absorb light (their opacity), and how the object changes over time.

**How you'll know whether to trust it, in three layers:**
1. **Maths checks** (Step 4): known exact answers, conservation, balance.
2. **Published models:** Ball et al. (2011), a full stellar-evolution code (Step 5), and Curtis et al. (2026), fits of 87 little red dots (Steps 9–12).
3. **Your measurements:** L ≈ 2.9 × 10⁴⁴ erg/s, T ≈ 4,970–5,500 K, R ≈ 1,400–1,700 AU (Steps 7–8).

**How to use:** run the grey code cells from top to bottom with **Shift + Enter**. The whole notebook runs in under a minute.

## Step 1 — Set up: physical constants and your measurements

This cell stores the constants of nature (in cgs units: centimetres, grams, seconds), the properties of the gas (70% hydrogen, 30% helium), your measurements of MoM-BH\*-1, and the plot colours. It prints the radius that goes with each of your two temperatures: expect **about 1,726 AU at 4,970 K and 1,410 AU at 5,500 K**.

This notebook holds the luminosity fixed at 2.9 × 10⁴⁴ erg/s. Your size notebook gave 1,304 AU at 5,500 K because there the luminosity changed with the temperature too (2.5 × 10⁴⁴ erg/s).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, trapezoid
from scipy.optimize import brentq, minimize_scalar

# --- constants of nature (cgs units) ---
G = 6.674e-8            # gravitational constant
c = 2.998e10            # speed of light
sigma_sb = 5.6704e-5    # Stefan-Boltzmann constant
a_rad = 7.5657e-15      # radiation constant
k_B = 1.380649e-16      # Boltzmann constant
m_u = 1.66054e-24       # atomic mass unit
Msun, Rsun, Lsun = 1.989e33, 6.957e10, 3.828e33
AU, YEAR = 1.496e13, 3.156e7

# --- the gas ---
X = 0.7                                  # hydrogen fraction by mass (the rest is helium)
MU = 1 / (2*X + 0.75*(1 - X))            # mean mass per particle of ionised gas (~0.62)
KAPPA_ES = 0.2 * (1 + X)                 # how strongly light pushes on ionised gas: 0.34 cm^2/g
GAMMA = 4/3                              # adiabatic index when radiation pressure dominates
n = 3                                    # polytropic index that goes with GAMMA = 4/3

# --- your measurements of MoM-BH*-1 ---
L_obs = 2.9e44                           # erg/s, from your blackbody fit
T_obs = (4970, 5500)                     # K: spectrum fit, and the MIRI/F444W colour
R_obs = [np.sqrt(L_obs / (4*np.pi*sigma_sb*T**4)) for T in T_obs]

print(f"Luminosity:  {L_obs:.2e} erg/s  =  {L_obs/Lsun:.2e} Suns")
for T, R in zip(T_obs, R_obs):
    print(f"Temperature: {T} K  ->  radius {R/AU:,.0f} AU")

# --- plot style and colours ---
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b", "axes.titlecolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 11, "legend.frameon": False, "legend.fontsize": 9.5})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"      # series colours, always in this order
INK2, MUTED, ZONE = "#52514e", "#898781", "#ebeae5"      # text, axis grey, "impossible" zones

# Part A — Stage 1: the balance check

## Step 2 — Could its own light blow it apart?

Light carries momentum. Inside the envelope the gas is hot enough to be fully ionised, and light pushes on its free electrons. For every gram of gas, the push is κL / (4πr²c), with κ = 0.34 cm²/g. Gravity pulls the same gram with GM / r². Both weaken as 1/r², so their ratio is the same at every radius:

**L / L_Edd**, where the Eddington luminosity is **L_Edd = 4πGcM / κ**.

If L / L_Edd is above 1, light beats gravity and the envelope is blown away. So the total mass must be at least **M_min = κL / (4πGc)**.

The cell also works out the **dynamical time**: roughly how fast the envelope would collapse if the light switched off, or expand if it got too bright. It tells you how quickly the balance has to be kept up.

**Expect:** a minimum mass of about **2.0 million Suns**, and a dynamical time of **about 8 years**.

In [ ]:
M_min = KAPPA_ES * L_obs / (4*np.pi*G*c)
print(f"Minimum total mass to stay together: {M_min/Msun:.2e} Suns")
print("(For comparison, the paper's black-hole estimate from line widths is 1e6 - 1e7 Suns.)\n")

for M in [M_min, 1e7*Msun]:
    t_dyn = np.sqrt(R_obs[0]**3 / (G*M))
    print(f"Dynamical time for {M/Msun:.1e} Suns inside {R_obs[0]/AU:,.0f} AU: {t_dyn/YEAR:.0f} years")

masses = np.logspace(5, 8, 300) * Msun
ratio = L_obs / (4*np.pi*G*c*masses/KAPPA_ES)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhspan(1, 100, color=ZONE, zorder=0)
ax.text(1.1e7, 5, "light beats gravity:\nthe envelope is blown away", color=INK2, fontsize=10, va="center")
ax.plot(masses/Msun, ratio, color=BLUE)
ax.axhline(1, color=MUTED, lw=1)
ax.axvline(M_min/Msun, color=MUTED, lw=1, ls="--")
ax.text(M_min/Msun*1.12, 0.015, f"minimum mass\n{M_min/Msun:.1e} Suns", color=INK2, fontsize=10)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1e5, 1e8); ax.set_ylim(0.01, 30)
ax.set_xlabel("Total mass of black hole + envelope (Suns)")
ax.set_ylabel("Light's push ÷ gravity's pull   (L / L_Edd)")
ax.set_title("MoM-BH*-1 must weigh at least ~2 million Suns to hold together", loc="left")
plt.tight_layout(); fig.savefig("step2_eddington.png", dpi=150); plt.show()

**What it means:** the black hole and its envelope together must weigh at least about 2 million Suns, or MoM-BH\*-1's own light would tear it apart. The balance is also delicate: if the black hole stopped shining, the envelope would start collapsing within about a decade of its own time. So the black hole has to keep feeding and shining continuously.

# Part B — Stage 2: the envelope model

## Step 3 — The equations, and the code that solves them

**In plain words**, the envelope obeys three rules at every radius r:
1. **Balance:** dP/dr = −G m(r) ρ / r² (the pressure gradient holds up the gas above it)
2. **Mass:** dm/dr = 4πr²ρ (each shell adds its mass)
3. **Pressure:** P = K ρ^(4/3) (radiation-dominated gas churned by convection)

Astronomers rewrite these in a unit-free form called the **Lane–Emden equation**. Write density as ρ = ρ₀θ³, radius as r = αξ, and mass as m = 4πρ₀α³φ. The three rules become just two:

**dθ/dξ = −φ/ξ²**  and  **dφ/dξ = ξ²θ³**

The code integrates them outward from the Bondi radius (ξ₀) until the density reaches zero (θ = 0). That's the surface (ξ₁).

**The black hole in the middle (a "loaded polytrope").** At the inner edge, φ already includes the black hole's mass plus the gas falling into it inside the Bondi radius. Following Ball et al. (2011), that falling gas has density ∝ r^(−3/2), so the starting values are:
- black hole: φ_BH = ξ₀ / 6 (this comes from the Bondi-radius formula);
- black hole + infalling gas: φ(ξ₀) = ξ₀/6 + (2/3)ξ₀³.

The black hole's share of the total mass is **q = φ_BH / φ_surface**.

**Turning it into real units.** Given the total mass, the black-hole mass and the luminosity, the code:
1. finds the Bondi radius ξ₀ that gives the right black-hole share q;
2. sets the pressure constant K from the total mass;
3. finds the density at the Bondi radius from the luminosity formula, L = (16πη/Γ) (GM_BH)² ρ₀ / c_s;
4. works out everything else from that: size, surface temperature, Bondi radius, inner temperature.

This cell takes a few seconds, because it pre-computes 300 envelopes.

In [ ]:
def lane_emden(xi, y):
    theta, phi = y
    return [-phi / xi**2, xi**2 * max(theta, 0.0)**n]

def at_surface(xi, y):
    return y[0]                               # the surface is where theta (density) reaches zero
at_surface.terminal, at_surface.direction = True, -1

def loaded_polytrope(xi0, dense=False):
    # Envelope around a black hole whose (unit-free) Bondi radius is xi0
    phi_bh = xi0 / (2*n)                          # the black hole
    phi_start = phi_bh + (2/3) * xi0**3           # + gas falling in inside the Bondi radius
    sol = solve_ivp(lane_emden, [xi0, 100], [1.0, phi_start], events=at_surface,
                    rtol=1e-9, atol=1e-12, dense_output=dense)
    return dict(xi0=xi0, phi_bh=phi_bh, xi1=sol.t_events[0][0],
                phi_surface=sol.y_events[0][0][1], sol=sol)

def bh_share(xi0):
    env = loaded_polytrope(xi0)
    return env["phi_bh"] / env["phi_surface"]

# the largest black-hole share that still allows a balanced envelope
best = minimize_scalar(lambda x: -bh_share(x), bounds=(0.8, 1.2), method="bounded",
                       options={"xatol": 1e-7})
XI0_MAX, Q_MAX = best.x, -best.fun

# pre-compute envelopes for black-hole shares from ~0 up to Q_MAX
_xi0 = np.geomspace(1e-7, XI0_MAX, 300)
_env = [loaded_polytrope(x) for x in _xi0]
_q = np.array([e["phi_bh"] / e["phi_surface"] for e in _env])
_phi_s = np.array([e["phi_surface"] for e in _env])
_xi1 = np.array([e["xi1"] for e in _env])

def envelope_shape(q):
    # Bondi radius, total (unit-free) mass and surface for black-hole share q
    lq = np.log(q)
    return (np.exp(np.interp(lq, np.log(_q), np.log(_xi0))),
            np.exp(np.interp(lq, np.log(_q), np.log(_phi_s))),
            np.interp(lq, np.log(_q), _xi1))

def temperature(P, rho):
    # the temperature at which gas pressure + radiation pressure add up to P
    return brentq(lambda T: rho*k_B*T/(MU*m_u) + a_rad*T**4/3 - P, 1.0, 1e12)

def quasi_star(M_total, M_bh, L, eta=0.1):
    # Balanced envelope for a total mass and black-hole mass (grams) and a luminosity (erg/s).
    # Returns None when no balanced envelope exists.
    q = M_bh / M_total
    if q >= Q_MAX:
        return None
    xi0, phi_s, xi1 = envelope_shape(q)
    K = np.pi*G * (M_total / (4*np.pi*phi_s))**(2/3)              # pressure constant, from the total mass
    A = (16*np.pi*eta/GAMMA) * (G*M_bh)**2 / np.sqrt(GAMMA*K)      # luminosity = A * rho0^(5/6)
    rho0 = (L / A)**(6/5)                                           # density at the Bondi radius
    alpha = np.sqrt(K/(np.pi*G)) * rho0**(-1/3)                     # length scale (cm)
    R = xi1 * alpha
    P0 = K * rho0**(4/3)
    T0 = temperature(P0, rho0)
    return dict(q=q, xi0=xi0, xi1=xi1, K=K, rho0=rho0, alpha=alpha, R=R, r_B=xi0*alpha,
                T_eff=(L / (4*np.pi*R**2*sigma_sb))**0.25, T0=T0,
                gas_share=rho0*k_B*T0/(MU*m_u)/P0,
                M_falling=4*np.pi*rho0*alpha**3*(2/3)*xi0**3,
                L=L, L_edd=4*np.pi*G*c*M_total/KAPPA_ES, M_total=M_total, M_bh=M_bh)

print(f"Ready. Largest black-hole share with a balanced envelope: {Q_MAX:.4f}")

## Step 4 — Layer 1: check the maths

Four tests. Each prints **PASS** or **FAIL**:
1. **A known exact answer.** With no black hole, the envelope must become an ordinary n = 3 polytrope: surface at ξ₁ = 6.89685, mass φ₁ = 2.01824, centre 54.18 times denser than average.
2. **The mass adds up.** For an example envelope (2 million Suns in total, a 10,000-Sun black hole, your luminosity), add up every layer, plus the black hole, plus the infalling gas. It must equal the total.
3. **Balance at every radius.** At 4,000 radii, compare the outward push of pressure (−dP/dr) with gravity's pull (Gmρ/r²). The push is estimated numerically from neighbouring radii, so expect a tiny mismatch, well under 1%.
4. **The Eddington limit.** The example's luminosity must not exceed its Eddington luminosity.

**If any test says FAIL, stop:** there's a bug, and nothing after this can be trusted.

In [ ]:
def check(name, ok, detail):
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {detail}")

# Test 1: no black hole -> ordinary n = 3 polytrope (start just off the centre with the series solution)
xs = 1e-4
sol = solve_ivp(lane_emden, [xs, 20], [1 - xs**2/6 + n*xs**4/120, xs**3/3 - n*xs**5/30],
                events=at_surface, rtol=1e-12, atol=1e-14)
xi1, phi1 = sol.t_events[0][0], sol.y_events[0][0][1]
check("known exact answer", abs(xi1 - 6.89685) < 1e-4 and abs(phi1 - 2.01824) < 1e-4,
      f"xi1 = {xi1:.5f} (6.89685), phi1 = {phi1:.5f} (2.01824), "
      f"centre/average density = {xi1**3/(3*phi1):.2f} (54.18)")

# The example envelope used in tests 2-4 (and in Step 6)
ex = quasi_star(2e6*Msun, 1e4*Msun, L_obs)
env = loaded_polytrope(ex["xi0"], dense=True)
xi = np.geomspace(ex["xi0"], env["xi1"], 4000)            # radii spaced evenly in log, dense near the black hole
theta, phi = env["sol"].sol(xi)
theta = np.clip(theta, 0, None)
r = ex["alpha"] * xi                                      # radius, cm
rho = ex["rho0"] * theta**n                               # density, g/cm^3
m = 4*np.pi*ex["rho0"]*ex["alpha"]**3 * phi               # mass inside r, g
P = ex["K"] * rho**(4/3)                                  # pressure

# Test 2: mass conservation
M_sum = ex["M_bh"] + ex["M_falling"] + trapezoid(4*np.pi*r**2*rho, r)
check("mass adds up", abs(M_sum/ex["M_total"] - 1) < 1e-3,
      f"black hole + infalling gas + envelope = {M_sum/Msun:,.0f} Suns (total: {ex['M_total']/Msun:,.0f})")

# Test 3: pressure balances gravity (skipping the thin outer skin where density is ~0)
push, pull = -np.gradient(P, r, edge_order=2), G * m * rho / r**2
inner = theta > 0.05
worst = np.max(np.abs(push[inner]/pull[inner] - 1))
check("pressure balances gravity", worst < 1e-2, f"worst mismatch {worst*100:.3f}% over {inner.sum()} radii")

# Test 4: the Eddington limit
check("light doesn't beat gravity", ex["L"] <= ex["L_edd"], f"L / L_Edd = {ex['L']/ex['L_edd']:.3f}")

## Step 5 — Layer 2: compare with Ball et al. (2011)

Ball et al. adapted a full stellar-evolution code (STARS) to follow a quasi-star of 10,000 Suns as its black hole grows. Their Table 1 lists the envelope at six moments. Three checks against it:

**5a. The black-hole limit.** They found that a balanced envelope exists only while the black hole holds less than **0.102** of the total mass, which happens when the Bondi radius is ξ₀ = 0.995. Your model should find the same.

The plot shows the black hole's share for every possible Bondi radius. Solutions to the right of the peak are considered unphysical, so the peak is the limit: past it, the envelope can't hold itself together.

In [ ]:
print(f"Your model:       largest black-hole share = {Q_MAX:.4f} at xi0 = {XI0_MAX:.3f}")
print(f"Ball et al. 2011: largest black-hole share = 0.102  at xi0 = 0.995")

xi_left = np.linspace(0.02, XI0_MAX, 120)
xi_right = np.linspace(XI0_MAX, 2.5, 80)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhspan(Q_MAX, 0.2, color=ZONE, zorder=0)
ax.text(0.04, 0.124, "no balanced envelope can hold a black hole this heavy", color=INK2, fontsize=10)
ax.plot(xi_left, [bh_share(x) for x in xi_left], color=BLUE, label="your model")
ax.plot(xi_right, [bh_share(x) for x in xi_right], color=BLUE, ls=":", lw=1.5)
ax.text(1.45, 0.087, "unphysical branch (dotted)", color=INK2, fontsize=10)
ax.plot(XI0_MAX, Q_MAX, "o", color=BLUE, ms=8)
ax.plot(0.995, 0.102, "x", color=ORANGE, ms=11, mew=2.5, label="Ball et al. 2011 (0.102)")
ax.set_xlim(0, 2.5); ax.set_ylim(0, 0.14)
ax.set_xlabel("Bondi radius (unit-free, ξ₀)")
ax.set_ylabel("Black hole's share of the total mass")
ax.set_title("Your model reproduces the published black-hole limit", loc="left")
ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig("step5a_bh_limit.png", dpi=150); plt.show()

**5b. The black hole's power.** Their table gives the density and the Bondi radius at each moment. Put those into the luminosity formula your model uses; it should give back their luminosity. This checks that you've implemented their prescription correctly. **Expect ratios of 0.98–1.02.**

In [ ]:
# Ball et al. (2011) Table 1, fiducial quasi-star (10,000 Suns, eta = 0.1):
# black hole (Suns), luminosity (1e8 Suns), density at r_B (g/cm^3), surface temperature (1e3 K),
# Bondi radius (100 R_sun), size (1e4 R_sun)
ball = np.array([
    [5,    3.48, 8.71e-5,  14.3, 0.0166, 0.303],
    [100,  2.92, 5.47e-8,  5.22, 3.66,   2.09],
    [200,  3.40, 1.30e-8,  4.77, 11.1,   2.70],
    [500,  4.83, 2.42e-9,  4.55, 41.2,   3.54],
    [1000, 6.05, 6.49e-10, 4.49, 113,    4.07],
    [1194, 5.81, 3.71e-10, 4.51, 185,    3.96]])
Mbh_b = ball[:, 0]
L_b, rho_b, T_b = ball[:, 1]*1e8*Lsun, ball[:, 2], ball[:, 3]*1e3
rB_b, R_b = ball[:, 4]*100*Rsun, ball[:, 5]*1e4*Rsun

print("black hole | luminosity from formula | luminosity in their table | ratio")
for Mbh, L_t, rho0, rB in zip(Mbh_b, L_b, rho_b, rB_b):
    cs = np.sqrt(2*G*Mbh*Msun / rB)                         # sound speed, from r_B = 2GM/cs^2
    L_formula = (16*np.pi*0.1/GAMMA) * (G*Mbh*Msun)**2 * rho0 / cs
    print(f"{Mbh:6.0f} Suns | {L_formula/Lsun:13.2e} Suns      | {L_t/Lsun:13.2e} Suns       | {L_formula/L_t:.3f}")

**5c. Size and temperature: the real test.** Give your model their total mass (10,000 Suns), black-hole mass and luminosity, and ask it to predict the rest. The table prints model ÷ Ball for each quantity (1.00 = perfect), and the plot shows three of them.

In [ ]:
print("black hole | share  | density at r_B | Bondi radius | size  | surface temperature (model, Ball)")
shares, dens_ratio, size_ratio, temp_ratio = [], [], [], []
for Mbh, L_t, rho0, T_t, rB, R in zip(Mbh_b, L_b, rho_b, T_b, rB_b, R_b):
    s = quasi_star(1e4*Msun, Mbh*Msun, L_t)
    if s is None:
        print(f"{Mbh:6.0f} Suns | {Mbh/1e4:.4f} | no balanced envelope: beyond your model's limit of {Q_MAX:.3f}")
        continue
    shares.append(Mbh/1e4); dens_ratio.append(s["rho0"]/rho0)
    size_ratio.append(s["R"]/R); temp_ratio.append(s["T_eff"]/T_t)
    print(f"{Mbh:6.0f} Suns | {Mbh/1e4:.4f} | {s['rho0']/rho0:8.2f}       | {s['r_B']/rB:8.2f}     | "
          f"{s['R']/R:5.2f} | {s['T_eff']:6.0f} K, {T_t:6.0f} K")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(1, color=MUTED, lw=1)
for y, col, lab in [(dens_ratio, AQUA, "density at Bondi radius"),
                    (size_ratio, ORANGE, "size of envelope"),
                    (temp_ratio, BLUE, "surface temperature")]:
    ax.plot(shares, y, "o-", color=col, ms=7, label=lab)
    ax.text(shares[-1]*1.15, y[-1], lab, color=INK2, fontsize=9.5, va="center")
ax.set_xscale("log")
ax.set_xlim(3e-4, 0.6); ax.set_ylim(0, 3)
ax.set_xlabel("Black hole's share of the total mass")
ax.set_ylabel("Your model ÷ Ball et al.  (1 = perfect)")
ax.set_title("The inside matches; the outer edge comes out too big", loc="left")
ax.legend(loc="upper left")
plt.tight_layout(); fig.savefig("step5c_vs_ball.png", dpi=150); plt.show()

**What 5c tells you (this is the most important validation result):**
- **The inside is right.** The density at the Bondi radius matches within about 20% (and the Bondi radius within about 15–45%).
- **The outer edge comes out too big**, by up to 2.7×. So at the same luminosity, the surface temperature comes out up to 40% too cool.
- **The error is smallest when the black hole is a small share of the total**: about 10% in temperature near a share of 0.005, which is where MoM-BH\*-1 turns out to be (Step 7).
- **Why:** the real envelope's outer layers are cooler and partly neutral, so they don't follow the n = 3 rule. Your model has no opacity physics there.
- **The last row** (1,194 Suns, a share of 0.119) has no solution in your model. Ball's full code went slightly past the polytrope limit (0.119 vs 0.102), which they also noted.

**What to trust:** the black-hole limit, completely. Inner quantities, to about 20–45%. Outer size and surface temperature, to about a factor of 2 in size and 10–40% in temperature, best when the black hole is a small share.

## Step 6 — How the envelope holds itself up

Now look inside one balanced envelope with MoM-BH\*-1's luminosity: 2 million Suns in total (about the minimum from Step 2) and a black hole of 10,000 Suns (Step 7 shows why). The cell prints its key numbers and plots density and temperature from the Bondi radius out to the surface. It also works out how much of the pressure holding the envelope up comes from **radiation** rather than hot gas.

**Expect:** a Bondi radius of about 12 AU, a size of about 1,740 AU, a surface near 4,950 K, about 400,000 K at the Bondi radius, and radiation providing over 99% of the pressure.

In [ ]:
sample = slice(None, None, 20)                                  # every 20th radius
T_r = np.array([temperature(Pi, ri) for Pi, ri in zip(P[sample], rho[sample])])
gas_share = rho[sample]*k_B*T_r/(MU*m_u) / P[sample]
inner_s = theta[sample] > 0.05

print(f"Total mass {ex['M_total']/Msun:,.0f} Suns, black hole {ex['M_bh']/Msun:,.0f} Suns (share {ex['q']:.4f})")
print(f"Bondi radius:           {ex['r_B']/AU:6.1f} AU")
print(f"Size (surface):         {ex['R']/AU:6.0f} AU      (your measurement: {R_obs[1]/AU:,.0f}-{R_obs[0]/AU:,.0f} AU)")
print(f"Surface temperature:    {ex['T_eff']:6.0f} K       (your measurement: {T_obs[0]}-{T_obs[1]} K)")
print(f"At the Bondi radius:    {ex['T0']:.2e} K, density {ex['rho0']:.1e} g/cm^3")
print(f"Pressure from radiation: {100*(1 - gas_share[inner_s].mean()):.1f}%  (hot gas provides the rest)")
print(f"L / L_Edd:               {ex['L']/ex['L_edd']:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, y, lab, col in [(ax1, rho[sample], "Density (g/cm³)", BLUE), (ax2, T_r, "Temperature (K)", ORANGE)]:
    ok = theta[sample] > 1e-4
    ax.plot(r[sample][ok]/AU, y[ok], color=col)
    ax.axvline(ex["r_B"]/AU, color=MUTED, lw=1, ls="--")
    ax.text(ex["r_B"]/AU*1.15, y[ok].min()*3, "Bondi\nradius", color=INK2, fontsize=9.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Distance from the black hole (AU)")
    ax.set_ylabel(lab)
ax1.set_title("Density inside the envelope", loc="left")
ax2.set_title("Temperature inside the envelope", loc="left")
plt.tight_layout(); fig.savefig("step6_structure.png", dpi=150); plt.show()

**How it sustains itself:** radiation provides over 99% of the pressure. The black hole's light, trapped in the envelope and slowly leaking out, is what holds the gas up; hot gas contributes less than 1%. That's also why the luminosity sits right at the Eddington limit (L / L_Edd ≈ 1). The light has to be strong enough to hold the envelope up, but no stronger, or it would blow the envelope away. The black hole keeps that light coming by swallowing gas at the Bondi radius, where the envelope is about 400,000 K.

## Step 7 — Which black hole fits MoM-BH\*-1?

**The key idea.** Your luminosity is fixed. A heavier black hole can make that luminosity from thinner gas at its Bondi radius. Thinner gas means a puffier envelope, and a puffier envelope sending out the same light has a cooler surface. So **your measured temperature points to a particular black-hole mass.**

The cell predicts the surface temperature for black holes from 100 Suns up to the limit, for a total mass of 2 million Suns and your luminosity, at three values of the uncertain efficiency η. Where each curve crosses your temperature band is the black-hole mass that fits.

**Expect:** about 4,500–18,000 Suns across the three efficiencies.

In [ ]:
M_total = M_min                                               # Step 2's minimum (~2 million Suns)
M_bh_max = 0.999 * Q_MAX * M_total / Msun                     # just below the black-hole limit
M_bh_grid = np.logspace(2, np.log10(M_bh_max), 200)
etas = [(0.1, BLUE), (0.03, ORANGE), (0.3, AQUA)]

def matching_bh(M_total, T, eta):
    # black-hole mass (Suns) whose envelope has surface temperature T
    f = lambda lm: quasi_star(M_total, 10**lm*Msun, L_obs, eta)["T_eff"] - T
    return 10**brentq(f, 1.0, np.log10(0.999*Q_MAX*M_total/Msun))

print(f"Total mass {M_total/Msun:.2e} Suns. Black-hole mass that matches your temperature:")
for eta, _ in etas:
    fits = [matching_bh(M_total, T, eta) for T in T_obs]
    print(f"  eta = {eta}: {min(fits):7,.0f} - {max(fits):7,.0f} Suns")
print(f"Largest black hole any balanced envelope of this mass can hold: {Q_MAX*M_total/Msun:,.0f} Suns")
print("The paper's estimate from line widths: 1,000,000 - 10,000,000 Suns")

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhspan(*T_obs, color="#cde2fb", zorder=0, label="your measured temperature")
ax.axvspan(M_bh_max, 1e7, color=ZONE, zorder=0)
ax.text(M_bh_max*1.15, 25000, "no balanced\nenvelope", color=INK2, fontsize=10)
for eta, col in etas:
    T_pred = [quasi_star(M_total, Mb*Msun, L_obs, eta)["T_eff"] for Mb in M_bh_grid]
    ax.plot(M_bh_grid, T_pred, color=col, label=f"model, efficiency η = {eta}")
ax.annotate("paper's estimate:\n1–10 million Suns →", xy=(1.9e6, 9000), ha="right", color=INK2, fontsize=10)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1e2, 2e6); ax.set_ylim(1500, 40000)
ax.set_yticks([2000, 3000, 5000, 10000, 20000, 40000])
ax.set_yticklabels(["2,000", "3,000", "5,000", "10,000", "20,000", "40,000"])
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlabel("Black-hole mass (Suns)")
ax.set_ylabel("Predicted surface temperature (K)")
ax.set_title("Your temperature points to a black hole of ~10,000 Suns", loc="left")
ax.legend(loc="lower left")
plt.tight_layout(); fig.savefig("step7_bh_mass.png", dpi=150); plt.show()

**What it means:** with η = 0.1, your temperature points to a black hole of about **8,000–10,000 Suns**; across η = 0.03–0.3, about 4,500–18,000 Suns. Step 5 showed the model runs roughly 10% too cool at this black-hole share; correcting for that would raise the answer by about 30%. A fair summary is **roughly 10,000 Suns, within a factor of about 2–3**.

(The curves turn up just before the limit because the envelope shrinks sharply there. They never come back up to your temperature, so there's no second solution.) That's **100–1,000 times lighter than the paper's estimate** of 1–10 million Suns, which came from line widths.

## Step 8 — The sustain map: which combinations can hold together?

One map pulls everything together. Total mass runs across, black-hole mass runs up.
- **Left grey zone:** too light. Its own light would blow it apart (Step 2).
- **Upper grey zone:** the black hole is too large a share. No balanced envelope exists (Step 5).
- **Blue band:** the black-hole masses that reproduce your temperature (Step 7), for η = 0.03–0.3 and T = 4,970–5,500 K.
- **Orange band:** the paper's black-hole estimate from line widths.

In [ ]:
Mt = np.logspace(np.log10(M_min/Msun), 8, 40) * Msun
lo, hi, mid = [], [], []
for M in Mt:
    fits = [matching_bh(M, T, eta) for eta, _ in etas for T in T_obs]
    lo.append(min(fits)); hi.append(max(fits)); mid.append(matching_bh(M, T_obs[0], 0.1))

x = np.logspace(5, 8, 100)
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.fill_between([1e5, M_min/Msun], 1e2, 1e8, color=ZONE, zorder=0, lw=0)
ax.fill_between(x, Q_MAX*x, 1e8, color=ZONE, zorder=0, lw=0)
ax.text(1.2e5, 3e2, "too light:\nits light\nblows it\napart", color=INK2, fontsize=10)
ax.text(1.5e5, 2.5e7, "black hole too large a share:\nno balanced envelope", color=INK2, fontsize=10)
ax.axhspan(1e6, 1e7, color=ORANGE, alpha=0.25, lw=0, label="paper's black-hole estimate (line widths)")
ax.fill_between(Mt/Msun, lo, hi, color=BLUE, alpha=0.3, lw=0, label="matches your temperature (η = 0.03–0.3)")
ax.plot(Mt/Msun, mid, color=BLUE, label="best match (η = 0.1, 4,970 K)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1e5, 1e8); ax.set_ylim(1e2, 1e8)
ax.set_xlabel("Total mass: black hole + envelope (Suns)")
ax.set_ylabel("Black-hole mass (Suns)")
ax.set_title("Where MoM-BH*-1 can hold itself together", loc="left")
ax.legend(loc="center right", bbox_to_anchor=(1, 0.55))
plt.tight_layout(); fig.savefig("step8_sustain_map.png", dpi=150); plt.show()

**Reading the map:** the blue band sits far below the orange band. If MoM-BH\*-1 is a classic quasi-star like Ball et al.'s, its black hole is about 100–1,000 times lighter than the line-width estimate. The blue band also barely moves with total mass, so this answer doesn't depend much on how heavy the envelope is.

The orange band only escapes the grey zones when the total mass is above about 10 million Suns. Even there, the model needs a far lighter black hole to match your temperature: a million-Sun black hole would give a much cooler surface than you measured.

There are two ways to read this:
1. **The line widths overestimate the mass.** The discovery paper itself says light scattering in the dense gas could inflate line-width masses by up to about 100 times, and Curtis et al. (2026) found no signs of overmassive black holes.
2. **MoM-BH\*-1 isn't a classic quasi-star.** Coughlin & Begelman (2024) treated the gas near the black hole differently and found balanced envelopes up to a black-hole share of about 60%. That would raise the upper grey zone and allow a much heavier black hole, but it would also need a different model to predict the temperature.

# Part C — Compare with Curtis et al. (2026)

## Step 9 — Load their public table

Curtis et al. fitted star-like atmosphere models to 87 little red dots, including MoM-BH\*-1, and published the results on GitHub. Each row has a temperature, a surface gravity (log g, in cm/s²), a luminosity (log of Suns) and an Eddington ratio φ, each with a low/middle/high estimate (16th/50th/84th percentiles). The cell keeps their main sample (the "primary" and "supplementary" fits), one row per object.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/o-curtis/lrd-quasistar-fits/main/summary/photospheric_parameters.csv"
table = pd.read_csv(URL)
main = table[table["family"].isin(["primary", "supplementary/prism", "supplementary/desi",
                                   "supplementary/grating"])]
main = main.drop_duplicates("target").reset_index(drop=True)
mom = main[main["target"] == "MoMBH1"].iloc[0]

print(f"{len(main)} little red dots loaded")
print(f"Temperatures: {main.teff_50.min():,.0f} - {main.teff_50.max():,.0f} K (median {main.teff_50.median():,.0f} K)")
print(f"\nMoM-BH*-1 in their fit: T = {mom.teff_50:,.0f} K, log g = {mom.logg_50:.2f}, "
      f"L = {10**mom.logL_50*Lsun:.2e} erg/s, phi = {mom.phi_50:.2f}")
print(f"MoM-BH*-1 in yours:     T = {T_obs[0]:,}-{T_obs[1]:,} K, L = {L_obs:.2e} erg/s")

## Step 10 — Layer 2: does your physics match theirs?

Their Eddington ratio is **φ = κσT⁴ / (g c)**: how hard light pushes on the surface gas compared with gravity, using the same κ = 0.34 cm²/g as your Step 2. Compute it yourself from their temperatures and gravities and compare with their φ column. If your physics matches theirs, the points sit on the diagonal line. **Expect a median ratio of about 1.00.**

In [ ]:
phi_mine = KAPPA_ES * sigma_sb * main.teff_50**4 / (10**main.logg_50 * c)
ratio = phi_mine / main.phi_50
print(f"your phi / their phi: median {ratio.median():.3f}  "
      f"(middle 68% of objects: {ratio.quantile(0.16):.2f} - {ratio.quantile(0.84):.2f})")

is_mom = main["target"] == "MoMBH1"
fig, ax = plt.subplots(figsize=(5.8, 5.6))
ax.plot([0.05, 1000], [0.05, 1000], color=MUTED, lw=1)
ax.plot(main.phi_50[~is_mom], phi_mine[~is_mom], "o", color=BLUE, ms=6, alpha=0.75,
        label="little red dots (Curtis et al.)")
ax.plot(main.phi_50[is_mom], phi_mine[is_mom], "o", color=ORANGE, ms=9, label="MoM-BH*-1")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(0.05, 1000); ax.set_ylim(0.05, 1000)
ax.set_xlabel("Their Eddington ratio φ")
ax.set_ylabel("Your Eddington ratio φ")
ax.set_title("Your formula reproduces theirs", loc="left")
ax.legend(loc="upper left")
plt.tight_layout(); fig.savefig("step10_phi_check.png", dpi=150); plt.show()

## Step 11 — Where MoM-BH\*-1 sits among little red dots

This is a **Hertzsprung–Russell diagram**: luminosity against temperature, hot on the left, as astronomers draw it. Dotted grey lines mark constant sizes of 100, 1,000 and 10,000 AU. Also shown: Ball et al.'s 10,000-Sun quasi-star as its black hole grows (from Step 5), and MoM-BH\*-1 from Curtis et al.'s fit and from yours.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
T_line = np.linspace(2500, 16000, 200)
for R_AU in [100, 1000, 10000]:
    L_line = 4*np.pi*(R_AU*AU)**2*sigma_sb*T_line**4 / Lsun
    ax.plot(T_line, L_line, color=MUTED, lw=1, ls=":")
    ax.text(2600, 4*np.pi*(R_AU*AU)**2*sigma_sb*2600**4/Lsun*1.5, f"{R_AU:,} AU",
            color=INK2, fontsize=9, ha="right")
ax.plot(main.teff_50[~is_mom], 10**main.logL_50[~is_mom], "o", color=BLUE, ms=6, alpha=0.7,
        label="little red dots (Curtis et al. 2026)")
ax.plot(T_b, L_b/Lsun, "o-", color=AQUA, ms=6, lw=1.5, label="Ball et al. quasi-star (10,000 Suns)")
ax.plot(mom.teff_50, 10**mom.logL_50, "s", color=ORANGE, ms=9, label="MoM-BH*-1, Curtis et al. fit")
ax.errorbar(np.mean(T_obs), L_obs/Lsun, xerr=(T_obs[1]-T_obs[0])/2, fmt="D", color=ORANGE, ms=8,
            mfc="white", mew=2, lw=2, label="MoM-BH*-1, your fit")
ax.set_yscale("log")
ax.set_xlim(16000, 2500); ax.set_ylim(1e6, 1e14)
ax.set_xlabel("Surface temperature (K)")
ax.set_ylabel("Luminosity (Suns)")
ax.set_title("Little red dots sit where quasi-stars sit, but much brighter", loc="left")
ax.legend(loc="lower left")
plt.tight_layout(); fig.savefig("step11_hr_diagram.png", dpi=150); plt.show()

**What to look for:**
- **Little red dots crowd into a narrow temperature range**, mostly about 3,500–5,500 K (median about 4,400 K). Ball et al.'s quasi-star settles at about 4,500 K once its black hole has grown. That's the "wall" quasi-stars can't cool past.
- **But most little red dots are about 10–300 times brighter** than Ball et al.'s 10,000-Sun quasi-star. As quasi-stars, they'd have to be roughly that much more massive, because quasi-stars shine near their Eddington limit and that limit grows in step with mass.
- **MoM-BH\*-1 is one of the two hottest objects in their sample** in their fit (about 6,500 K). Your fit is cooler (about 5,000–5,500 K) and dimmer: their atmosphere model puts more light in ultraviolet wavelengths that weren't observed.

## Step 12 — Weighing MoM-BH\*-1 two independent ways

1. **Your Eddington minimum (Step 2):** the least mass that stops its light from blowing it apart.
2. **Curtis et al.'s surface gravity:** mass = g R² / G, using their gravity and the radius from their luminosity and temperature.

If MoM-BH\*-1 is held together near the balance point, these two should agree. **Expect** about 2.4 million Suns from their gravity (low–high: about 0.35–6.9 million), against your 2.0 million.

In [ ]:
R_c = np.sqrt(10**mom.logL_50*Lsun / (4*np.pi*sigma_sb*mom.teff_50**4))
print(f"Radius from their fit: {R_c/AU:,.0f} AU\n")
for p, word in [("16", "low "), ("50", "middle"), ("84", "high")]:
    M_g = 10**mom[f"logg_{p}"] * R_c**2 / G
    print(f"Mass from their surface gravity ({word}, log g = {mom[f'logg_{p}']:.2f}): {M_g/Msun:.1e} Suns")
print(f"\nYour Eddington minimum, with your luminosity:  {M_min/Msun:.1e} Suns")
print(f"Your Eddington minimum, with their luminosity: {KAPPA_ES*10**mom.logL_50*Lsun/(4*np.pi*G*c)/Msun:.1e} Suns")
print(f"\nMedian phi of their sample: {main.phi_50.median():.1f}   (MoM-BH*-1: {mom.phi_50:.2f})")
print(f"Objects closer to the balance point (smaller phi) than MoM-BH*-1: "
      f"{(main.phi_50 < mom.phi_50).mean()*100:.0f}% of their sample")

**What it means:** two independent methods agree. Your Eddington minimum (from your luminosity) and Curtis et al.'s surface gravity (from fitting the shape of the spectrum) both give about 2 million Suns. MoM-BH\*-1 sits close to the balance point, as you'd expect for an envelope held up by its own light.

Most of their sample has φ well above 1 (median about 11). Be careful interpreting that: φ assumes fully ionised gas (κ = 0.34), but at 4,000–6,500 K the surface gas is mostly neutral, and light pushes on it much less. So φ above 1 doesn't by itself mean those objects are flying apart. Interpreting φ is part of Curtis et al.'s paper.

# What the simulator tells you

**How MoM-BH\*-1 can hold itself together:**
- **Its light is what holds it up.** Radiation provides over 99% of the pressure, so the envelope sits right at the Eddington limit (Steps 2 and 6).
- **It must weigh at least about 2 million Suns** in total, or its own light would blow it apart. Curtis et al.'s surface gravity independently gives about 2.4 million (Steps 2 and 12).
- **A classic quasi-star can hold together only while its black hole is below about 10% of the total.** For MoM-BH\*-1, that means under roughly 200,000 Suns. This limit reproduces Ball et al. (2011) to within 0.5% (Step 5a).
- **Your measured temperature points to a black hole of roughly 10,000 Suns** (within a factor of about 2–3) in a classic quasi-star. That's 100–1,000 times below the line-width estimate (Steps 7–8).

**How far to trust it:** the maths passes every check (Step 4). Against a full stellar-evolution code, the black-hole limit and the inner structure match well. The outer size is too big by up to 2.7×, which makes the surface temperature up to 40% too cool, though only about 10% at MoM-BH\*-1's black-hole share (Step 5c). The efficiency η is uncertain by at least a factor of 10. The model also leaves out rotation, magnetic fields, winds, surface opacity and time evolution.

**For a write-up, cite:** Ball et al. 2011 (MNRAS 414, 2751) for the model and the comparison table; Coughlin & Begelman 2024 (ApJ 970, 158) for the alternative ~60% limit; Curtis et al. 2026 (arXiv:2609.09265) for the little-red-dot fits; and Naidu et al. 2026 (Nature) for MoM-BH\*-1.

**Next stages:**
- **Stage 3, evolution:** let the black hole eat the envelope over time. How long can MoM-BH\*-1 stay this way, and does that fit how common black hole stars are?
- **Stage 4, AI:** run this model many thousands of times, train a neural network to imitate it, and use it to find the full range of masses and efficiencies that fit all your measurements together.